In [12]:
import os
import torch
import torchvision
from torchvision.datasets import CelebA
from torch.utils.data import DataLoader, Subset, random_split
import numpy as np
import matplotlib.pyplot as plt
#from tdqm import tdqm
import torchvision.transforms as transforms

In [13]:
if torch.mps.is_available:
    device = torch.device("mps")
    print("using mps")
elif torch.cuda.is_available:
    device = torch.device("cuda")
    print('using cuda')
else:
    device = torch.device("cpu")
    print("using cpu")

using mps


In [20]:
#Hyperparameters
BATCH_SIZE = 128
IMG_SIZE = 64
CHANNELS = 3
LATENT_DIM = 100

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
print("transforms ready")

transforms ready


In [22]:
#Load datasets and apply transformations
celeba_dir = '../'
dataset = CelebA(
    root = celeba_dir,
    split = "train",
    download = False,
    transform=transform
)

#split data into training and validaiton
dataset_size = len(dataset)
train_size = int(0.9*dataset_size)
val_size = dataset_size - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

#create dataloaders
train_loader = DataLoader(train_dataset, 
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          num_workers=0
                         )
val_loader = DataLoader(val_dataset,
                        batch_size=BATCH_SIZE,
                        shuffle=False,
                        num_workers=0
                       )

#Check data shape
sample_img, _ = next(iter(train_loader))
print(f"Image batch shape: {sample_img.shape}")
print(f"Image value range: [{sample_img.min():.2f}, {sample_img.max():.2f}]")



Image batch shape: torch.Size([128, 3, 64, 64])
Image value range: [-1.00, 1.00]


Vanilla Gan Implementation

In [ ]:
#Vanilla Gan architecture
import torch.nn as nn

# --------------------------
# Generator (Vanilla GAN)
# --------------------------
class VanillaGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, channels=CHANNELS):
        super(VanillaGenerator, self).__init__()
        self.main = nn.Sequential(
            # Step 1: FC layer → (512,4,4) (per lab)
            nn.Linear(latent_dim, 512 * 4 * 4),
            nn.BatchNorm1d(512 * 4 * 4),
            nn.ReLU(inplace=True),
            nn.Unflatten(1, (512, 4, 4)),  # Reshape to (512,4,4)
            
            # Transposed Conv: Upscale to (256,8,8)
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            
            # Transposed Conv: Upscale to (128,16,16)
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            # Transposed Conv: Upscale to (64,32,32)
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            
            # Transposed Conv: Upscale to (3,64,64) (final output)
            nn.ConvTranspose2d(64, channels, 4, 2, 1, bias=False),
            nn.Tanh()  # Output [-1,1] (matches data normalization)
        )

    def forward(self, x):
        return self.main(x)

# --------------------------
# Discriminator (Vanilla GAN)
# --------------------------
class VanillaDiscriminator(nn.Module):
    def __init__(self, channels=CHANNELS):
        super(VanillaDiscriminator, self).__init__()
        self.main = nn.Sequential(
            # Conv: (3,64,64) → (64,32,32)
            nn.Conv2d(channels, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),  # slope=0.2 (per lab)
            
            # Conv: (64,32,32) → (128,16,16)
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Conv: (128,16,16) → (256,8,8)
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Conv: (256,8,8) → (512,4,4)
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Flatten → Scalar + Sigmoid (0-1 for real/fake)
            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x)

# Initialize models
gen_vanilla = VanillaGenerator().to(device)
dis_vanilla = VanillaDiscriminator().to(device)

# Test model shapes
z = torch.randn(1, LATENT_DIM).to(device)  # Random latent vector
fake_img = gen_vanilla(z)
dis_score = dis_vanilla(fake_img)
print(f"Generator output shape: {fake_img.shape}")  # (1,3,64,64) ✔️
print(f"Discriminator output shape: {dis_score.shape}")  # (1,1) ✔️

In [ ]:
#Vanilla Gan training
# Loss function: Binary Cross Entropy (matches minimax loss for Vanilla GAN)
criterion = nn.BCELoss()

# Optimizers (per lab)
lr = 3e-4
beta1 = 0.5
beta2 = 0.999
optimizer_G = torch.optim.Adam(gen_vanilla.parameters(), lr=lr, betas=(beta1, beta2))
optimizer_D = torch.optim.Adam(dis_vanilla.parameters(), lr=lr, betas=(beta1, beta2))

# Hyperparameters for training
EPOCHS = 5
FIXED_NOISE = torch.randn(64, LATENT_DIM).to(device)  # For fixed visualization (8x8 grid)

In [ ]:
#Vanilla Gan training loop
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# Create folder for saved images
os.makedirs("vanilla_gan_images", exist_ok=True)
os.makedirs("loss_plots", exist_ok=True)

# Loss tracking
vanilla_G_losses = []
vanilla_D_losses = []

# Training loop
for epoch in range(EPOCHS):
    gen_vanilla.train()
    dis_vanilla.train()
    epoch_G_loss = 0.0
    epoch_D_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for real_imgs, _ in pbar:
        batch_size = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        
        # --------------------------
        # Train Discriminator: max log(D(x)) + log(1-D(G(z)))
        # --------------------------
        optimizer_D.zero_grad()
        # Label real: 1, fake: 0 (with small label smoothing for stability: 0.9 instead of 1)
        label_real = torch.full((batch_size, 1), 0.9, device=device)
        label_fake = torch.full((batch_size, 1), 0.0, device=device)
        
        # Score real images
        output_real = dis_vanilla(real_imgs)
        loss_D_real = criterion(output_real, label_real)
        loss_D_real.backward()
        
        # Score fake images (generate & detach to avoid training G)
        noise = torch.randn(batch_size, LATENT_DIM, device=device)
        fake_imgs = gen_vanilla(noise)
        output_fake = dis_vanilla(fake_imgs.detach())
        loss_D_fake = criterion(output_fake, label_fake)
        loss_D_fake.backward()
        
        # Total D loss + optimize
        loss_D = loss_D_real + loss_D_fake
        optimizer_D.step()
        
        # --------------------------
        # Train Generator: min log(1-D(G(z))) → max log(D(G(z)))
        # --------------------------
        optimizer_G.zero_grad()
        # Reuse fake images, label as real (fool D)
        output_fake_G = dis_vanilla(fake_imgs)
        loss_G = criterion(output_fake_G, label_real)
        loss_G.backward()
        optimizer_G.step()
        
        # Accumulate losses
        epoch_G_loss += loss_G.item()
        epoch_D_loss += loss_D.item()
        pbar.set_postfix({"G Loss": loss_G.item(), "D Loss": loss_D.item()})
    
    # Average epoch losses
    avg_G_loss = epoch_G_loss / len(train_loader)
    avg_D_loss = epoch_D_loss / len(train_loader)
    vanilla_G_losses.append(avg_G_loss)
    vanilla_D_losses.append(avg_D_loss)
    print(f"Epoch {epoch+1} | Avg G Loss: {avg_G_loss:.4f} | Avg D Loss: {avg_D_loss:.4f}")
    
    # Visualize generated images every 5 epochs (per lab)
    if (epoch+1) % 5 == 0 or epoch == 0:
        gen_vanilla.eval()
        with torch.no_grad():
            fixed_fake = gen_vanilla(FIXED_NOISE).detach().cpu()
        # Normalize back to [0,1] for plotting
        fixed_fake = (fixed_fake + 1) / 2.0
        # Save 8x8 grid
        grid = torchvision.utils.make_grid(fixed_fake, nrow=8, padding=2)
        plt.figure(figsize=(10,10))
        plt.axis("off")
        plt.imshow(np.transpose(grid, (1,2,0)))
        plt.title(f"Vanilla GAN - Epoch {epoch+1}")
        plt.savefig(f"vanilla_gan_images/epoch_{epoch+1}.png")
        plt.close()

# Save models
torch.save(gen_vanilla.state_dict(), "vanilla_generator.pth")
torch.save(dis_vanilla.state_dict(), "vanilla_discriminator.pth")

Task 3 WasserStein GAN

In [ ]:
# --------------------------
# WGAN Critic (no Sigmoid!)
# --------------------------
class WGANCritic(nn.Module):
    def __init__(self, channels=CHANNELS):
        super(WGANCritic, self).__init__()
        self.main = nn.Sequential(
            # Exact same conv layers as Vanilla Discriminator (no changes)
            nn.Conv2d(channels, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 1)  # NO SIGMOID (real-valued score)
        )

    def forward(self, x):
        return self.main(x)

# Initialize WGAN models (Generator = Vanilla Generator, Critic = new WGAN Critic)
gen_wgan = VanillaGenerator().to(device)
critic_wgan = WGANCritic().to(device)

# Test shapes
fake_img_wgan = gen_wgan(z)
critic_score = critic_wgan(fake_img_wgan)
print(f"WGAN Critic output shape: {critic_score.shape}")  # (1,1) ✔️ (no Sigmoid)

In [ ]:
# WGAN Hyperparameters (per lab)
CRITIC_ITERATIONS = 5  # Train critic 5x for 1 G update
WEIGHT_CLIP = 0.01     # Clip weights to [-0.01, 0.01]

# Optimizers (Adam, 3e-4; RMSProp also works)
optimizer_G_wgan = torch.optim.Adam(gen_wgan.parameters(), lr=lr, betas=(beta1, beta2))
optimizer_C_wgan = torch.optim.Adam(critic_wgan.parameters(), lr=lr, betas=(beta1, beta2))

# Fixed noise for visualization (8x8 grid)
FIXED_NOISE_WGAN = torch.randn(64, LATENT_DIM).to(device)

# Create folder for WGAN images
os.makedirs("wgan_images", exist_ok=True)

# Loss tracking
wgan_G_losses = []
wgan_C_losses = []

In [ ]:
# WGAN Training Loop
for epoch in range(EPOCHS):
    gen_wgan.train()
    critic_wgan.train()
    epoch_G_loss = 0.0
    epoch_C_loss = 0.0
    pbar = tqdm(train_loader, desc=f"WGAN Epoch {epoch+1}/{EPOCHS}")
    
    for i, (real_imgs, _) in enumerate(pbar):
        batch_size = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        critic_wgan.zero_grad()
        
        # --------------------------
        # Train Critic (CRITIC_ITERATIONS times per G update)
        # --------------------------
        for _ in range(CRITIC_ITERATIONS):
            # Score real images
            output_real = critic_wgan(real_imgs).reshape(-1)
            # Score fake images
            noise = torch.randn(batch_size, LATENT_DIM, device=device)
            fake_imgs = gen_wgan(noise)
            output_fake = critic_wgan(fake_imgs.detach()).reshape(-1)
            
            # WGAN Critic Loss: E(D(x)) - E(D(G(z))) (maximize → minimize negative)
            loss_C = -(torch.mean(output_real) - torch.mean(output_fake))
            loss_C.backward()
            optimizer_C_wgan.step()
            
            # Weight Clipping (per lab: [-0.01, 0.01])
            for p in critic_wgan.parameters():
                p.data.clamp_(-WEIGHT_CLIP, WEIGHT_CLIP)
        
        # --------------------------
        # Train Generator (1x per 5 critic updates)
        # --------------------------
        optimizer_G_wgan.zero_grad()
        # WGAN Generator Loss: -E(D(G(z))) (minimize)
        output_fake_G = critic_wgan(fake_imgs).reshape(-1)
        loss_G = -torch.mean(output_fake_G)
        loss_G.backward()
        optimizer_G_wgan.step()
        
        # Accumulate losses
        epoch_G_loss += loss_G.item()
        epoch_C_loss += loss_C.item()
        pbar.set_postfix({"G Loss": loss_G.item(), "C Loss": loss_C.item()})
    
    # Average epoch losses
    avg_G_loss = epoch_G_loss / len(train_loader)
    avg_C_loss = epoch_C_loss / len(train_loader)
    wgan_G_losses.append(avg_G_loss)
    wgan_C_losses.append(avg_C_loss)
    print(f"WGAN Epoch {epoch+1} | Avg G Loss: {avg_G_loss:.4f} | Avg C Loss: {avg_C_loss:.4f}")
    
    # Visualize every 5 epochs (per lab)
    if (epoch+1) % 5 == 0 or epoch == 0:
        gen_wgan.eval()
        with torch.no_grad():
            fixed_fake = gen_wgan(FIXED_NOISE_WGAN).detach().cpu()
        fixed_fake = (fixed_fake + 1) / 2.0
        grid = torchvision.utils.make_grid(fixed_fake, nrow=8, padding=2)
        plt.figure(figsize=(10,10))
        plt.axis("off")
        plt.imshow(np.transpose(grid, (1,2,0)))
        plt.title(f"WGAN - Epoch {epoch+1}")
        plt.savefig(f"wgan_images/epoch_{epoch+1}.png")
        plt.close()

# Save WGAN models
torch.save(gen_wgan.state_dict(), "wgan_generator.pth")
torch.save(critic_wgan.state_dict(), "wgan_critic.pth")

Evaluation

In [ ]:
# Plot Loss Curves
plt.figure(figsize=(16, 8))

# Vanilla GAN Loss
plt.subplot(1, 2, 1)
plt.plot(vanilla_G_losses, label="Generator Loss", color="blue")
plt.plot(vanilla_D_losses, label="Discriminator Loss", color="red")
plt.title("Vanilla GAN - Loss Curves")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

# WGAN Loss
plt.subplot(1, 2, 2)
plt.plot(wgan_G_losses, label="Generator Loss", color="blue")
plt.plot(wgan_C_losses, label="Critic Loss", color="red")
plt.title("WGAN - Loss Curves")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("loss_plots/gan_wgan_loss_curves.png")
plt.close()